# State-Conditioned Multi-Objective BC Screen v1

Purpose: continue from `warm_start_official_failure_handoff_v1`, where the best official warm-start diagnostic (`trend_plain_contract`) still had a joint trade-off: `wf_2022` relative return was high but MaxDD exceeded 15%, while `wf_2023_2024` had acceptable drawdown but only +8.22pp relative return. This notebook keeps the special experiment logic in runtime and tests a fold-local training target that combines:

- reentry participation floor for high-trend / lower-stress states, derived only from each train fold's feature distribution;
- drawdown/risk cap for high-stress or deep-drawdown states;
- official warm-start qlib replay sliced back to the requested evaluation window.

Governance:

- Baselines: canonical `v19reg_v22_budget_usage_bull_guard_v1`; direct comparison `sequence_trend_joint_warm_start_official_eval_v1` and `warm_start_official_failure_handoff_v1`.
- Role: diagnose / training-target screen, not promotion.
- Target windows: `wf_2022`, `wf_2023_2024`.
- Acceptance: all evaluated folds need `relative_return >= +30pp` and `MaxDD <= 15%`; otherwise no promotion and no actor retention.
- Changed layer: `model_selection`, `training_target`, `behavior_cloning_warm_start`.
- External references: Qlib modular quant platform (`https://github.com/microsoft/qlib`, `https://arxiv.org/abs/2009.11189`), FinRL risk-aware trading workflow (`https://github.com/AI4Finance-Foundation/FinRL`, `https://arxiv.org/abs/2011.09607`), DAgger / BC distribution shift risk (`https://proceedings.mlr.press/v15/ross11a.html`), and multi-objective sequential decision framing (`https://arxiv.org/abs/1312.4077`).
- Closest success: `trend_participation_bc_warmstart_screen_v1` and `sequence_trend_joint_bc_screen_v1` showed participation targets can improve cold-split gates.
- Closest failure: `sequence_trend_joint_warm_start_official_eval_v1` showed the same family failed official warm-start gates, so this notebook must use official replay metrics.
- Why not duplicate: the labels here are generated from fold-local feature quantiles and combine both reentry participation and drawdown-cap states in the same target contract, instead of replaying the previous static trend/sequence contracts.


In [1]:

from __future__ import annotations

import json
import math
import os
import random
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "AGENTS.md").exists():
    ROOT = Path("/Users/rziiiii/.openclaw/workspace")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.state_conditioned_objective import daily_path_metrics
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.core.metric_reconciliation import (
    build_warm_start_replay_segment,
    slice_warm_start_daily_path,
)
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, flatten_feature_frame, init_qlib
from src.soxl_quant.policy.policy_builder import state_objective_target_position_from_row
from src.soxl_quant.settings import LOOKBACK, QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = "state_conditioned_multi_objective_bc_screen_v1"
CARD = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
FRAME_PATH = ROOT / "data" / "soxl_profile_balanced_rule_residual_v22.pkl"
PROFILE_NAME = "sparse_release_bc_ppo_screen_v1"
BASE_TARGETS_PATH = ROOT / "runtime/model_selection/card/cap10_sequence_risk_target_cap_bc_screen_v1/baseline_no_sequence_cap_target_contract.csv"
WARM_OFFICIAL_CARD = ROOT / "runtime/model_selection/card/sequence_trend_joint_warm_start_official_eval_v1"
HANDOFF_CARD = ROOT / "runtime/model_selection/card/warm_start_official_failure_handoff_v1"

SEED = 20260526
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
BC_TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=300, lr=0.001, batch_size=128, seed=SEED)
WALK_FORWARD_WINDOWS = {
    "wf_2022": {"target_fold": "wf_2022_train", "train": ("2010-03-11", "2020-12-31"), "valid": ("2021-01-01", "2021-12-31"), "test": ("2022-01-01", "2022-12-31")},
    "wf_2023_2024": {"target_fold": "wf_2023_2024_train", "train": ("2010-03-11", "2022-12-31"), "valid": ("2023-01-01", "2023-12-31"), "test": ("2024-01-01", "2024-12-31")},
}
EXTERNAL_SOURCES = [
    "https://github.com/microsoft/qlib",
    "https://arxiv.org/abs/2009.11189",
    "https://github.com/AI4Finance-Foundation/FinRL",
    "https://arxiv.org/abs/2011.09607",
    "https://proceedings.mlr.press/v15/ross11a.html",
    "https://arxiv.org/abs/1312.4077",
]

CANDIDATES = [
    {
        "candidate": "multi_obj_floor45_cap20_riskfirst",
        "floor": 0.45,
        "max_reentry_position": 0.60,
        "max_teacher_position": 0.36,
        "risk_cap": 0.20,
        "reentry_bonus": 6.0,
        "risk_bonus": 12.0,
        "sample_weight_config": StateObjectiveBCSampleWeightConfig(
            rebound_multiplier=2.5,
            downtrend_multiplier=3.5,
            ordinary_multiplier=1.0,
            objective_weight_power=0.15,
            class_balanced_beta=0.99,
            weight_cap=25.0,
        ),
    },
    {
        "candidate": "multi_obj_floor55_cap25_balanced",
        "floor": 0.55,
        "max_reentry_position": 0.70,
        "max_teacher_position": 0.40,
        "risk_cap": 0.25,
        "reentry_bonus": 10.0,
        "risk_bonus": 10.0,
        "sample_weight_config": StateObjectiveBCSampleWeightConfig(
            rebound_multiplier=4.0,
            downtrend_multiplier=3.0,
            ordinary_multiplier=1.0,
            objective_weight_power=0.20,
            class_balanced_beta=0.99,
            weight_cap=30.0,
        ),
    },
    {
        "candidate": "multi_obj_floor65_cap20_weighted",
        "floor": 0.65,
        "max_reentry_position": 0.80,
        "max_teacher_position": 0.42,
        "risk_cap": 0.20,
        "reentry_bonus": 14.0,
        "risk_bonus": 14.0,
        "sample_weight_config": StateObjectiveBCSampleWeightConfig(
            rebound_multiplier=5.0,
            downtrend_multiplier=4.0,
            ordinary_multiplier=1.0,
            objective_weight_power=0.25,
            class_balanced_beta=0.99,
            weight_cap=35.0,
        ),
    },
]
FEATURE_COLS = [
    "anchor_gap_180",
    "trend_gap_63",
    "trend_slope_63",
    "mom_63",
    "realized_vol_63",
    "drawdown_63",
    "vix_percentile_252",
    "tracking_error_vol_21",
    "relative_strength_21",
    "regime_risk",
    "regime_risk_no_vix",
    "regime_risk_velocity_5",
]


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, np.floating):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def to_bool_series(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False).astype(bool)
    return series.fillna(False).map(lambda x: str(x).strip().lower() in {"1", "true", "yes", "y"} if isinstance(x, str) else bool(x)).astype(bool)


def load_targets(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path, parse_dates=["datetime"])
    bool_cols = [
        "is_rebound_cluster",
        "is_downtrend_state",
        "release_gate_state",
        "drawdown_control_state",
        "sequence_risk_state",
        "consensus_reentry_state",
        "consensus_drawdown_cap_state",
    ]
    for col in bool_cols:
        if col not in frame.columns:
            frame[col] = False
        frame[col] = to_bool_series(frame[col])
    numeric_cols = [
        "teacher_position",
        "rebound_target_position",
        "risk_target_position",
        "risk_target_position_original",
        "objective_weight",
        "rebound_participation_weight",
    ]
    for col in numeric_cols:
        if col not in frame.columns:
            frame[col] = frame["risk_target_position"] if col == "risk_target_position_original" and "risk_target_position" in frame.columns else 0.0
        frame[col] = pd.to_numeric(frame[col], errors="coerce").fillna(0.0)
    if "target_recipe" not in frame.columns:
        frame["target_recipe"] = frame.get("candidate", "")
    if "source_components" not in frame.columns:
        frame["source_components"] = frame.get("candidate", "")
    return frame.sort_values(["fold", "datetime"]).reset_index(drop=True)


def fold_subset(targets: pd.DataFrame, spec: dict[str, Any]) -> pd.DataFrame:
    dates = pd.to_datetime(targets["datetime"]).dt.normalize()
    return targets.loc[
        targets["fold"].astype(str).eq(str(spec["target_fold"]))
        & (dates >= pd.Timestamp(spec["train"][0]).normalize())
        & (dates <= pd.Timestamp(spec["train"][1]).normalize())
    ].sort_values("datetime").reset_index(drop=True)


def gate_loss(relative_return: float, max_dd: float) -> float:
    return float(max(0.0, TARGET_RELATIVE_RETURN - relative_return) + 10.0 * max(0.0, max_dd - TARGET_MAX_DD))


def quantiles(frame: pd.DataFrame) -> dict[str, float]:
    qspec = {
        "anchor_gap_180_q55": ("anchor_gap_180", 0.55),
        "trend_gap_63_q55": ("trend_gap_63", 0.55),
        "trend_slope_63_q55": ("trend_slope_63", 0.55),
        "mom_63_q60": ("mom_63", 0.60),
        "mom_63_q40": ("mom_63", 0.40),
        "realized_vol_63_q70": ("realized_vol_63", 0.70),
        "realized_vol_63_q80": ("realized_vol_63", 0.80),
        "drawdown_63_q45": ("drawdown_63", 0.45),
        "drawdown_63_q20": ("drawdown_63", 0.20),
        "vix_percentile_252_q55": ("vix_percentile_252", 0.55),
        "vix_percentile_252_q80": ("vix_percentile_252", 0.80),
        "regime_risk_q80": ("regime_risk", 0.80),
        "regime_risk_no_vix_q80": ("regime_risk_no_vix", 0.80),
        "tracking_error_vol_21_q80": ("tracking_error_vol_21", 0.80),
        "relative_strength_21_q45": ("relative_strength_21", 0.45),
    }
    out: dict[str, float] = {}
    for name, (col, q) in qspec.items():
        out[name] = float(pd.to_numeric(frame[col], errors="coerce").dropna().quantile(q))
    return out


def append_component(value: object, component: str) -> str:
    existing = str(value) if value is not None and str(value) != "nan" else ""
    if not component:
        return existing
    parts = [p for p in existing.split(",") if p]
    if component not in parts:
        parts.append(component)
    return ",".join(parts)


def build_multi_objective_targets(base_targets: pd.DataFrame, train_frame: pd.DataFrame, cfg: dict[str, Any]) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    feature_frame = train_frame.reset_index()[["datetime", *FEATURE_COLS]].copy()
    feature_frame["datetime"] = pd.to_datetime(feature_frame["datetime"]).dt.normalize()
    target_rows: list[pd.DataFrame] = []
    audit_rows: list[dict[str, Any]] = []
    threshold_rows: list[dict[str, Any]] = []
    candidate = str(cfg["candidate"])
    for fold_name, spec in WALK_FORWARD_WINDOWS.items():
        train_start = pd.Timestamp(spec["train"][0]).normalize()
        train_end = pd.Timestamp(spec["train"][1]).normalize()
        fold_base = fold_subset(base_targets, spec).copy()
        train_features = feature_frame.loc[(feature_frame["datetime"] >= train_start) & (feature_frame["datetime"] <= train_end)].copy()
        if fold_base.empty:
            audit_rows.append({"candidate": candidate, "fold": fold_name, "target_rows": 0, "reason": "empty_fold_base"})
            continue
        qs = quantiles(train_features)
        threshold_rows.append({"candidate": candidate, "fold": fold_name, **qs})
        merged = fold_base.drop(columns=[col for col in FEATURE_COLS if col in fold_base.columns]).copy()
        merged["datetime"] = pd.to_datetime(merged["datetime"]).dt.normalize()
        merged = merged.merge(feature_frame, on="datetime", how="left")
        missing_feature_rows = int(merged[FEATURE_COLS].isna().all(axis=1).sum())
        for col in ["teacher_position", "rebound_target_position", "risk_target_position", "risk_target_position_original", "objective_weight", "rebound_participation_weight"]:
            merged[col] = pd.to_numeric(merged[col], errors="coerce").fillna(0.0).clip(lower=0.0)
        for col in ["is_rebound_cluster", "is_downtrend_state", "drawdown_control_state", "release_gate_state", "sequence_risk_state"]:
            merged[col] = to_bool_series(merged[col])

        reentry = (
            (merged["mom_63"] >= qs["mom_63_q60"])
            & (merged["trend_slope_63"] >= qs["trend_slope_63_q55"])
            & (merged["trend_gap_63"] >= qs["trend_gap_63_q55"])
            & (merged["anchor_gap_180"] >= qs["anchor_gap_180_q55"])
            & (merged["vix_percentile_252"] <= qs["vix_percentile_252_q55"])
            & (merged["realized_vol_63"] <= qs["realized_vol_63_q70"])
            & (merged["drawdown_63"] >= qs["drawdown_63_q45"])
            & (merged["relative_strength_21"] >= qs["relative_strength_21_q45"])
            & (merged["teacher_position"] <= float(cfg["max_teacher_position"]))
        )
        risk = (
            ((merged["drawdown_63"] <= qs["drawdown_63_q20"]) & (merged["trend_slope_63"] <= qs["trend_slope_63_q55"]))
            | ((merged["vix_percentile_252"] >= qs["vix_percentile_252_q80"]) & (merged["realized_vol_63"] >= qs["realized_vol_63_q80"]))
            | ((merged["regime_risk"] >= qs["regime_risk_q80"]) & (merged["mom_63"] <= qs["mom_63_q40"]))
            | ((merged["regime_risk_no_vix"] >= qs["regime_risk_no_vix_q80"]) & (merged["tracking_error_vol_21"] >= qs["tracking_error_vol_21_q80"]))
        ) & (merged["teacher_position"] > float(cfg["risk_cap"]) + 0.02)
        reentry = reentry & ~risk

        floor_position = np.minimum(
            float(cfg["max_reentry_position"]),
            np.maximum(merged["teacher_position"].to_numpy(dtype=float), float(cfg["floor"])),
        )
        merged["candidate"] = candidate
        merged["target_recipe"] = candidate
        merged["consensus_reentry_state"] = reentry.astype(bool)
        merged["consensus_drawdown_cap_state"] = risk.astype(bool)
        merged["state_conditioned_multi_objective_state"] = (reentry | risk).astype(bool)
        merged["is_rebound_cluster"] = merged["is_rebound_cluster"] | reentry
        merged["is_downtrend_state"] = merged["is_downtrend_state"] | risk
        merged["drawdown_control_state"] = merged["drawdown_control_state"] | risk
        merged["rebound_target_position"] = np.where(
            reentry,
            np.maximum(merged["rebound_target_position"].to_numpy(dtype=float), floor_position),
            merged["rebound_target_position"].to_numpy(dtype=float),
        )
        merged["risk_target_position_original"] = np.where(
            risk,
            merged["risk_target_position"].to_numpy(dtype=float),
            merged["risk_target_position_original"].to_numpy(dtype=float),
        )
        merged["risk_target_position"] = np.where(
            risk,
            np.minimum(merged["risk_target_position"].to_numpy(dtype=float), float(cfg["risk_cap"])),
            merged["risk_target_position"].to_numpy(dtype=float),
        )
        merged["rebound_participation_weight"] = np.where(
            reentry,
            np.maximum(merged["rebound_participation_weight"].to_numpy(dtype=float), float(cfg["reentry_bonus"])),
            merged["rebound_participation_weight"].to_numpy(dtype=float),
        )
        merged["objective_weight"] = merged["objective_weight"] + np.where(reentry, float(cfg["reentry_bonus"]), 0.0) + np.where(risk, float(cfg["risk_bonus"]), 0.0)
        merged["source_components"] = [append_component(v, "consensus_reentry_floor" if is_reentry else "") for v, is_reentry in zip(merged["source_components"], reentry)]
        merged["source_components"] = [append_component(v, "consensus_drawdown_cap" if is_risk else "") for v, is_risk in zip(merged["source_components"], risk)]
        merged["state_conditioned_target_position"] = [state_objective_target_position_from_row(row) for row in merged.to_dict("records")]
        merged["feature_threshold_recipe"] = "fold_local_quantile_consensus_reentry_plus_drawdown_cap"
        target_rows.append(merged)
        audit_rows.append({
            "candidate": candidate,
            "fold": fold_name,
            "target_rows": int(len(merged)),
            "missing_feature_rows": missing_feature_rows,
            "reentry_rows": int(reentry.sum()),
            "risk_cap_rows": int(risk.sum()),
            "state_rows": int((reentry | risk).sum()),
            "mean_teacher_position": float(merged["teacher_position"].mean()),
            "mean_state_target_position": float(merged["state_conditioned_target_position"].mean()),
            "mean_reentry_target_position": float(merged.loc[reentry, "state_conditioned_target_position"].mean()) if bool(reentry.any()) else math.nan,
            "mean_risk_target_position": float(merged.loc[risk, "state_conditioned_target_position"].mean()) if bool(risk.any()) else math.nan,
        })
    all_targets = pd.concat(target_rows, ignore_index=True) if target_rows else pd.DataFrame()
    if not all_targets.empty:
        all_targets = all_targets.sort_values(["fold", "datetime"]).reset_index(drop=True)
    return all_targets, pd.DataFrame(audit_rows), pd.DataFrame(threshold_rows)

base_targets = load_targets(BASE_TARGETS_PATH)
train_frame = pd.read_pickle(FRAME_PATH)
profile = load_profile(PROFILE_NAME)
init_qlib(QLIB_DATA_PATH)
feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
prior_official = pd.read_csv(WARM_OFFICIAL_CARD / "warm_start_official_metrics.csv")
handoff_selection = json.loads((HANDOFF_CARD / "selection.json").read_text(encoding="utf-8"))
print("handoff", handoff_selection.get("decision"), handoff_selection.get("next_mechanism"))
print("train_frame", train_frame.shape, str(train_frame.index.min().date()), str(train_frame.index.max().date()))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[92385:MainThread](2026-05-26 18:02:55,103) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[92385:MainThread](2026-05-26 18:02:55,924) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[92385:MainThread](2026-05-26 18:02:55,926) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


handoff train_state_conditioned_multi_objective train_state_conditioned_multi_objective
train_frame (2973, 64) 2011-03-11 2022-12-30


In [2]:

targets_by_candidate: dict[str, pd.DataFrame] = {}
audit_frames: list[pd.DataFrame] = []
threshold_frames: list[pd.DataFrame] = []
for cfg in CANDIDATES:
    targets, audit, thresholds = build_multi_objective_targets(base_targets, train_frame, cfg)
    targets_by_candidate[str(cfg["candidate"])] = targets
    targets.to_csv(CARD / f"{cfg['candidate']}_target_contract.csv", index=False)
    audit_frames.append(audit)
    threshold_frames.append(thresholds)

target_audit = pd.concat(audit_frames, ignore_index=True) if audit_frames else pd.DataFrame()
threshold_audit = pd.concat(threshold_frames, ignore_index=True) if threshold_frames else pd.DataFrame()
target_audit.to_csv(CARD / "target_state_audit.csv", index=False)
threshold_audit.to_csv(CARD / "fold_feature_thresholds.csv", index=False)
config_rows = []
for cfg in CANDIDATES:
    row = {k: v for k, v in cfg.items() if k != "sample_weight_config"}
    row["sample_weight_config"] = json.dumps(asdict(cfg["sample_weight_config"]), sort_keys=True)
    config_rows.append(row)
pd.DataFrame(config_rows).to_csv(CARD / "candidate_configs.csv", index=False)
display(target_audit)
display(threshold_audit)


,candidate,fold,target_rows,missing_feature_rows,reentry_rows,risk_cap_rows,state_rows,mean_teacher_position,mean_state_target_position,mean_reentry_target_position,mean_risk_target_position
0,multi_obj_floor45_cap20_riskfirst,wf_2022,253,0,20,66,86,0.462764,0.369896,0.45,0.191866
1,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,756,0,34,65,99,0.297715,0.264485,0.45,0.191741
2,multi_obj_floor55_cap25_balanced,wf_2022,253,0,20,66,86,0.462764,0.389658,0.55,0.237320
3,multi_obj_floor55_cap25_balanced,wf_2023_2024,756,0,34,64,98,0.297715,0.272884,0.55,0.236924
4,multi_obj_floor65_cap20_weighted,wf_2022,253,0,20,66,86,0.462764,0.385706,0.65,0.191866
5,multi_obj_floor65_cap20_weighted,wf_2023_2024,756,0,34,65,99,0.297715,0.273480,0.65,0.191741


,candidate,fold,anchor_gap_180_q55,trend_gap_63_q55,trend_slope_63_q55,mom_63_q60,mom_63_q40,realized_vol_63_q70,realized_vol_63_q80,drawdown_63_q45,drawdown_63_q20,vix_percentile_252_q55,vix_percentile_252_q80,regime_risk_q80,regime_risk_no_vix_q80,tracking_error_vol_21_q80,relative_strength_21_q45
0,multi_obj_floor45_cap20_riskfirst,wf_2022,0.217722,0.063328,0.061668,0.257826,0.135672,0.805250,0.854766,-0.124128,-0.277913,0.523810,0.781746,0.590079,0.573654,0.078065,0.312862
1,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,0.202196,0.058498,0.057430,0.247982,0.117794,0.843034,0.965539,-0.142289,-0.319217,0.521825,0.789683,0.625333,0.620380,0.077580,0.287669
2,multi_obj_floor55_cap25_balanced,wf_2022,0.217722,0.063328,0.061668,0.257826,0.135672,0.805250,0.854766,-0.124128,-0.277913,0.523810,0.781746,0.590079,0.573654,0.078065,0.312862
3,multi_obj_floor55_cap25_balanced,wf_2023_2024,0.202196,0.058498,0.057430,0.247982,0.117794,0.843034,0.965539,-0.142289,-0.319217,0.521825,0.789683,0.625333,0.620380,0.077580,0.287669
4,multi_obj_floor65_cap20_weighted,wf_2022,0.217722,0.063328,0.061668,0.257826,0.135672,0.805250,0.854766,-0.124128,-0.277913,0.523810,0.781746,0.590079,0.573654,0.078065,0.312862
5,multi_obj_floor65_cap20_weighted,wf_2023_2024,0.202196,0.058498,0.057430,0.247982,0.117794,0.843034,0.965539,-0.142289,-0.319217,0.521825,0.789683,0.625333,0.620380,0.077580,0.287669


In [3]:

# Build fold-local full feature frames once. Official replay includes warmup rows;
# metrics are sliced/rebased to the requested test window only.
fold_cache: dict[str, dict[str, Any]] = {}
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    dataset = build_dataset(
        train_seg=spec["train"],
        valid_seg=spec["valid"],
        test_seg=spec["test"],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    feature_full = flatten_feature_frame(dataset.handler.feature_frame).droplevel("instrument").copy()
    price_full = dataset.handler.price_frame.droplevel("instrument")["price"].sort_index()
    warm_seg = build_warm_start_replay_segment(feature_full.index, eval_seg=spec["test"], lookback=LOOKBACK)
    fold_cache[fold_name] = {"dataset": dataset, "feature_full": feature_full, "price_full": price_full, "warm_seg": warm_seg}

pd.DataFrame([
    {
        "fold": fold,
        "feature_start": str(cache["feature_full"].index.min().date()),
        "feature_end": str(cache["feature_full"].index.max().date()),
        "test_start": WALK_FORWARD_WINDOWS[fold]["test"][0],
        "test_end": WALK_FORWARD_WINDOWS[fold]["test"][1],
        "warm_start": cache["warm_seg"][0],
        "warm_end": cache["warm_seg"][1],
    }
    for fold, cache in fold_cache.items()
]).to_csv(CARD / "warm_start_segments.csv", index=False)

metric_rows: list[dict[str, Any]] = []
fit_rows: list[dict[str, Any]] = []
slice_rows: list[pd.DataFrame] = []
action_rows: list[pd.DataFrame] = []
daily_manifest: list[dict[str, str]] = []

for cfg_idx, cfg in enumerate(CANDIDATES):
    candidate = str(cfg["candidate"])
    targets = targets_by_candidate[candidate]
    for fold_name, spec in WALK_FORWARD_WINDOWS.items():
        fold_targets = fold_subset(targets, spec)
        if fold_targets.empty:
            metric_rows.append({"candidate": candidate, "fold": fold_name, "status": "skipped_no_targets"})
            continue
        set_seed(SEED + cfg_idx * 17 + len(fold_name))
        fit = fit_state_objective_bc_actor_from_frame(
            train_frame,
            targets=fold_targets,
            profile=profile,
            train_config=BC_TRAIN_CONFIG,
            sample_weight_config=cfg["sample_weight_config"],
            sample_size=0,
            rng_seed=SEED,
        )
        fit_rows.append({
            "candidate": candidate,
            "fold": fold_name,
            "target_rows": int(len(fold_targets)),
            "bc_samples": int(len(fit.actions)),
            "before_accuracy": float(fit.metrics.get("before_accuracy", np.nan)),
            "after_accuracy": float(fit.metrics.get("after_accuracy", np.nan)),
            "after_loss": float(fit.metrics.get("after_loss", np.nan)),
            "weight_mean": float(fit.metrics.get("weight_mean", np.nan)) if "weight_mean" in fit.metrics else np.nan,
            "weight_max": float(fit.metrics.get("weight_max", np.nan)) if "weight_max" in fit.metrics else np.nan,
        })
        bc_dataset = build_state_objective_bc_dataset_from_frame(
            train_frame,
            targets=fold_targets,
            profile=profile,
            sample_size=0,
            rng_seed=SEED,
        )
        sm = evaluate_bc_actor_slice_fit(
            fit.actor,
            fit.obs,
            fit.actions,
            bc_dataset.metadata,
            slice_cols=(
                "is_rebound_cluster",
                "is_downtrend_state",
                "consensus_reentry_state",
                "consensus_drawdown_cap_state",
                "state_conditioned_multi_objective_state",
            ),
        )
        sm.insert(0, "fold", fold_name)
        sm.insert(0, "candidate", candidate)
        slice_rows.append(sm)
        dist = fit.action_distribution.copy()
        dist.insert(0, "fold", fold_name)
        dist.insert(0, "candidate", candidate)
        action_rows.append(dist)

        cache = fold_cache[fold_name]
        bt = backtest_profile(
            fit.actor,
            cache["feature_full"],
            cache["price_full"],
            position_cap_config=profile.position_cap,
            position_budget_config=profile.position_budget,
            execution_config=profile.execution,
            asset_idx=0,
            feature_columns=feature_columns,
            model_config=profile.model,
            mode_router_config=profile.mode_router,
            test_seg=cache["warm_seg"],
        )
        warm_daily = slice_warm_start_daily_path(
            portfolio_df=bt["portfolio"],
            price_series=cache["price_full"],
            eval_seg=spec["test"],
            position_history=bt.get("position_history", []),
            trade_signals=bt.get("trade_signals", []),
        )
        daily_path = CARD / f"{candidate}_{fold_name}_warm_start_official_daily_path.csv"
        warm_daily.to_csv(daily_path, index=False)
        metrics = daily_path_metrics(warm_daily)
        prior_ref = prior_official[
            prior_official["candidate"].astype(str).eq("trend_plain_contract")
            & prior_official["fold"].astype(str).eq(fold_name)
        ]
        prior_rel = float(prior_ref["official_warm_relative_return"].iloc[0]) if not prior_ref.empty else math.nan
        prior_dd = float(prior_ref["official_warm_max_dd"].iloc[0]) if not prior_ref.empty else math.nan
        metric_rows.append({
            "candidate": candidate,
            "fold": fold_name,
            "status": "evaluated",
            "requested_start": spec["test"][0],
            "requested_end": spec["test"][1],
            "warm_start": cache["warm_seg"][0],
            "warm_end": cache["warm_seg"][1],
            "official_warm_return": float(metrics["return"]),
            "official_warm_buyhold_return": float(metrics["buy_and_hold_return"]),
            "official_warm_relative_return": float(metrics["relative_return"]),
            "official_warm_max_dd": float(metrics["max_dd"]),
            "official_warm_avg_pos": float(metrics["avg_pos"]),
            "trades": int(bt.get("trades", 0)),
            "full_gate": bool(float(metrics["relative_return"]) >= TARGET_RELATIVE_RETURN and float(metrics["max_dd"]) <= TARGET_MAX_DD),
            "gate_loss": gate_loss(float(metrics["relative_return"]), float(metrics["max_dd"])),
            "relative_delta_vs_trend_plain_official": float(metrics["relative_return"] - prior_rel) if math.isfinite(prior_rel) else math.nan,
            "max_dd_delta_vs_trend_plain_official": float(metrics["max_dd"] - prior_dd) if math.isfinite(prior_dd) else math.nan,
            "daily_path": str(daily_path.relative_to(ROOT)),
        })
        daily_manifest.append({"candidate": candidate, "fold": fold_name, "path": str(daily_path.relative_to(ROOT))})

fit_metrics = pd.DataFrame(fit_rows).sort_values(["candidate", "fold"]).reset_index(drop=True)
bc_slice_metrics = pd.concat(slice_rows, ignore_index=True) if slice_rows else pd.DataFrame()
action_distribution = pd.concat(action_rows, ignore_index=True) if action_rows else pd.DataFrame()
warm_metrics = pd.DataFrame(metric_rows).sort_values(["candidate", "fold"]).reset_index(drop=True)
fit_metrics.to_csv(CARD / "bc_fit_metrics.csv", index=False)
bc_slice_metrics.to_csv(CARD / "bc_slice_metrics.csv", index=False)
action_distribution.to_csv(CARD / "bc_action_distribution.csv", index=False)
warm_metrics.to_csv(CARD / "warm_start_official_metrics.csv", index=False)
pd.DataFrame(daily_manifest).to_csv(CARD / "daily_manifest.csv", index=False)
display(fit_metrics)
display(bc_slice_metrics)
display(warm_metrics)


[92385:MainThread](2026-05-26 18:02:58,204) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[92385:MainThread](2026-05-26 18:02:58,205) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[92385:MainThread](2026-05-26 18:02:58,206) INFO - qlib.timer - [log.py:127] - Time cost: 0.003s | Init data Done


[92385:MainThread](2026-05-26 18:03:00,647) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[92385:MainThread](2026-05-26 18:03:00,648) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


...

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

,candidate,fold,target_rows,bc_samples,before_accuracy,after_accuracy,after_loss,weight_mean,weight_max
0,multi_obj_floor45_cap20_riskfirst,wf_2022,253,253,0.458498,0.881423,1.025630,1.0,2.490636
1,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,756,755,0.329801,0.765563,1.144428,1.0,2.088680
2,multi_obj_floor55_cap25_balanced,wf_2022,253,253,0.371542,0.897233,1.009241,1.0,2.308388
3,multi_obj_floor55_cap25_balanced,wf_2023_2024,756,755,0.299338,0.758940,1.149197,1.0,3.379651
4,multi_obj_floor65_cap20_weighted,wf_2022,253,253,0.454545,0.857708,1.051574,1.0,2.421810
5,multi_obj_floor65_cap20_weighted,wf_2023_2024,756,755,0.328477,0.729801,1.177985,1.0,2.747951


,candidate,fold,slice,samples,accuracy,loss,label_unique_actions,predicted_unique_actions,label_dominant_action,label_dominant_share,predicted_dominant_action,predicted_dominant_share
0,multi_obj_floor45_cap20_riskfirst,wf_2022,all,253,0.881423,1.025630,5,5,4,0.391304,4,0.399209
1,multi_obj_floor45_cap20_riskfirst,wf_2022,consensus_drawdown_cap_state,66,0.984848,0.920644,3,3,3,0.651515,3,0.636364
2,multi_obj_floor45_cap20_riskfirst,wf_2022,consensus_reentry_state,20,1.000000,0.907956,2,2,4,0.850000,4,0.850000
3,multi_obj_floor45_cap20_riskfirst,wf_2022,is_downtrend_state,122,0.975410,0.930675,4,5,3,0.352459,1,0.344262
4,multi_obj_floor45_cap20_riskfirst,wf_2022,is_rebound_cluster,86,0.988372,0.917839,5,5,3,0.500000,3,0.488372
5,multi_obj_floor45_cap20_riskfirst,wf_2022,state_conditioned_multi_objective_state,86,0.988372,0.917693,5,5,3,0.500000,3,0.488372
6,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,all,755,0.765563,1.144428,5,5,1,0.292715,2,0.251656
7,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,consensus_drawdown_cap_state,65,0.846154,1.066363,3,4,3,0.661538,3,0.692308
8,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,consensus_reentry_state,34,0.941176,0.967373,2,2,4,0.911765,4,0.970588
9,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,is_downtrend_state,480,0.835417,1.076279,4,5,1,0.397917,2,0.350000


,candidate,fold,status,requested_start,requested_end,warm_start,warm_end,official_warm_return,official_warm_buyhold_return,official_warm_relative_return,official_warm_max_dd,official_warm_avg_pos,trades,full_gate,gate_loss,relative_delta_vs_trend_plain_official,max_dd_delta_vs_trend_plain_official,daily_path
0,multi_obj_floor45_cap20_riskfirst,wf_2022,evaluated,2022-01-01,2022-12-31,2021-10-07,2022-12-30,-27.200043,-86.588072,59.388029,30.626799,0.164765,41,False,156.267987,-8.510789,10.427842,runtime/model_selection/card/state_conditioned...
1,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,evaluated,2024-01-01,2024-12-31,2023-10-05,2024-12-31,29.905886,-0.356635,30.262521,16.122029,0.221759,49,False,11.220292,22.037958,3.836302,runtime/model_selection/card/state_conditioned...
2,multi_obj_floor55_cap25_balanced,wf_2022,evaluated,2022-01-01,2022-12-31,2021-10-07,2022-12-30,-30.762538,-86.588072,55.825533,31.808525,0.162252,43,False,168.085252,-12.073284,11.609568,runtime/model_selection/card/state_conditioned...
3,multi_obj_floor55_cap25_balanced,wf_2023_2024,evaluated,2024-01-01,2024-12-31,2023-10-05,2024-12-31,7.700831,-0.356635,8.057465,19.036155,0.209318,48,False,62.304085,-0.167097,6.750427,runtime/model_selection/card/state_conditioned...
4,multi_obj_floor65_cap20_weighted,wf_2022,evaluated,2022-01-01,2022-12-31,2021-10-07,2022-12-30,-30.166149,-86.588072,56.421922,31.567204,0.174270,45,False,165.672038,-11.476895,11.368247,runtime/model_selection/card/state_conditioned...
5,multi_obj_floor65_cap20_weighted,wf_2023_2024,evaluated,2024-01-01,2024-12-31,2023-10-05,2024-12-31,22.262959,-0.356635,22.619594,18.949796,0.244511,49,False,46.878366,14.395031,6.664068,runtime/model_selection/card/state_conditioned...


In [4]:

evaluated = warm_metrics[warm_metrics["status"].eq("evaluated")].copy()
summary_rows: list[dict[str, Any]] = []
for candidate, group in evaluated.groupby("candidate", sort=True):
    group = group.sort_values("fold")
    row: dict[str, Any] = {
        "candidate": candidate,
        "fold_count": int(len(group)),
        "full_gate_all": bool(group["full_gate"].all()),
        "full_gate_count": int(group["full_gate"].sum()),
        "gate_loss": float(group["gate_loss"].sum()),
        "min_relative_return": float(group["official_warm_relative_return"].min()),
        "max_max_dd": float(group["official_warm_max_dd"].max()),
        "mean_avg_pos": float(group["official_warm_avg_pos"].mean()),
        "total_trades": int(group["trades"].sum()),
        "mean_relative_delta_vs_trend_plain_official": float(group["relative_delta_vs_trend_plain_official"].mean()),
        "max_dd_delta_vs_trend_plain_official": float(group["max_dd_delta_vs_trend_plain_official"].max()),
    }
    for fold in WALK_FORWARD_WINDOWS:
        sub = group[group["fold"].eq(fold)]
        if not sub.empty:
            rec = sub.iloc[0]
            row[f"{fold}_relative_return"] = float(rec["official_warm_relative_return"])
            row[f"{fold}_max_dd"] = float(rec["official_warm_max_dd"])
            row[f"{fold}_avg_pos"] = float(rec["official_warm_avg_pos"])
            row[f"{fold}_full_gate"] = bool(rec["full_gate"])
    summary_rows.append(row)

candidate_summary = pd.DataFrame(summary_rows).sort_values(
    ["full_gate_all", "full_gate_count", "gate_loss", "candidate"],
    ascending=[False, False, True, True],
).reset_index(drop=True)
candidate_summary.to_csv(CARD / "candidate_summary.csv", index=False)
selected = candidate_summary.iloc[0].to_dict() if not candidate_summary.empty else {}
selected_candidate = str(selected.get("candidate", ""))
full_gate_candidate_count = int(candidate_summary["full_gate_all"].sum()) if not candidate_summary.empty else 0

if full_gate_candidate_count > 0:
    decision = "state_conditioned_multi_objective_full_gate_candidate_no_promotion"
    next_mechanism = "run_fixed_split_wf2020_actor_retention_and_checkpoint_review_before_any_promotion"
else:
    best_rel = float(selected.get("min_relative_return", math.nan)) if selected else math.nan
    best_dd = float(selected.get("max_max_dd", math.nan)) if selected else math.nan
    if math.isfinite(best_rel) and best_rel < TARGET_RELATIVE_RETURN and math.isfinite(best_dd) and best_dd > TARGET_MAX_DD:
        decision = "state_conditioned_multi_objective_joint_tradeoff_no_promotion"
        next_mechanism = "do_not_extend_same_bc_grid; move to model architecture or reward terms that separate reentry from drawdown state"
    elif math.isfinite(best_rel) and best_rel < TARGET_RELATIVE_RETURN:
        decision = "state_conditioned_multi_objective_underparticipation_no_promotion"
        next_mechanism = "inspect reentry label coverage and consider state head or PPO only after BC slice fit is adequate"
    else:
        decision = "state_conditioned_multi_objective_drawdown_failed_no_promotion"
        next_mechanism = "revise risk-state representation or checkpoint scoring before increasing reentry floor"

prior_reference = prior_official[prior_official["candidate"].astype(str).eq("trend_plain_contract")].copy()
prior_reference.to_csv(CARD / "trend_plain_official_reference_metrics.csv", index=False)

selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "role": "diagnose",
    "profile": PROFILE_NAME,
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": ["sequence_trend_joint_warm_start_official_eval_v1", "warm_start_official_failure_handoff_v1"],
    "closest_historical_success_case": "sequence_trend_joint_bc_screen_v1 improved cold-split gates with composed targets but failed official warm-start replay",
    "closest_historical_failure_case": "sequence_trend_joint_warm_start_official_eval_v1 had zero full-gate candidates under official warm-start metrics",
    "why_not_duplicate": "This notebook builds new fold-local feature-quantile target contracts with both consensus reentry floors and drawdown caps, then evaluates them through the official warm-start replay gate.",
    "changed_layer": ["model_selection", "training_target", "behavior_cloning_warm_start"],
    "target_relative_return_pp": TARGET_RELATIVE_RETURN,
    "target_max_dd_pct": TARGET_MAX_DD,
    "folds_evaluated": list(WALK_FORWARD_WINDOWS.keys()),
    "candidates_evaluated": [cfg["candidate"] for cfg in CANDIDATES],
    "selected_candidate": selected_candidate,
    "selected_summary": selected,
    "full_gate_candidate_count": full_gate_candidate_count,
    "promotion_allowed": False,
    "ppo_screen_allowed_next": bool(full_gate_candidate_count > 0),
    "training_allowed_from_this_card": bool(full_gate_candidate_count > 0),
    "actor_retention": "no actor bundle retained; diagnostic BC actors are regenerated by this notebook",
    "handoff_source_decision": handoff_selection.get("decision"),
    "handoff_next_mechanism": handoff_selection.get("next_mechanism"),
    "external_sources": EXTERNAL_SOURCES,
    "next_mechanism": next_mechanism,
    "sidecars": [
        "candidate_configs.csv",
        "fold_feature_thresholds.csv",
        "target_state_audit.csv",
        "*_target_contract.csv",
        "warm_start_segments.csv",
        "bc_fit_metrics.csv",
        "bc_slice_metrics.csv",
        "bc_action_distribution.csv",
        "warm_start_official_metrics.csv",
        "trend_plain_official_reference_metrics.csv",
        "candidate_summary.csv",
        "daily_manifest.csv",
        "*_warm_start_official_daily_path.csv",
        "selection.json",
        "summary.json",
    ],
}
summary = {
    **selection,
    "candidate_summary": candidate_summary.to_dict("records"),
    "warm_start_official_metrics": warm_metrics.to_dict("records"),
    "target_state_audit": target_audit.to_dict("records"),
    "conclusion": [],
}
for row in candidate_summary.to_dict("records"):
    summary["conclusion"].append(
        f"{row['candidate']}: min official relative={row['min_relative_return']:.2f}pp, "
        f"max MaxDD={row['max_max_dd']:.2f}%, full_gate_all={bool(row['full_gate_all'])}, "
        f"gate_loss={row['gate_loss']:.2f}."
    )
if full_gate_candidate_count == 0 and selected:
    summary["conclusion"].append(
        "No candidate clears the +30pp / <=15% official warm-start gate across both folds; no promotion, no actor retention, and no long PPO from this card."
    )
summary["conclusion"].append(f"Next action: {next_mechanism}.")

write_json(CARD / "selection.json", selection)
write_json(CARD / "summary.json", summary)
display(candidate_summary)
selection


,candidate,fold_count,full_gate_all,full_gate_count,gate_loss,min_relative_return,max_max_dd,mean_avg_pos,total_trades,mean_relative_delta_vs_trend_plain_official,max_dd_delta_vs_trend_plain_official,wf_2022_relative_return,wf_2022_max_dd,wf_2022_avg_pos,wf_2022_full_gate,wf_2023_2024_relative_return,wf_2023_2024_max_dd,wf_2023_2024_avg_pos,wf_2023_2024_full_gate
0,multi_obj_floor45_cap20_riskfirst,2,False,0,167.488279,30.262521,30.626799,0.193262,90,6.763585,10.427842,59.388029,30.626799,0.164765,False,30.262521,16.122029,0.221759,False
1,multi_obj_floor65_cap20_weighted,2,False,0,212.550404,22.619594,31.567204,0.209391,94,1.459068,11.368247,56.421922,31.567204,0.174270,False,22.619594,18.949796,0.244511,False
2,multi_obj_floor55_cap25_balanced,2,False,0,230.389336,8.057465,31.808525,0.185785,91,-6.120191,11.609568,55.825533,31.808525,0.162252,False,8.057465,19.036155,0.209318,False


{'experiment': 'state_conditioned_multi_objective_bc_screen_v1',
 'created_at': '2026-05-26T10:04:04.432654+00:00',
 'decision': 'state_conditioned_multi_objective_drawdown_failed_no_promotion',
 'role': 'diagnose',
 'profile': 'sparse_release_bc_ppo_screen_v1',
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'direct_comparison_baseline': ['sequence_trend_joint_warm_start_official_eval_v1',
  'warm_start_official_failure_handoff_v1'],
 'closest_historical_success_case': 'sequence_trend_joint_bc_screen_v1 improved cold-split gates with composed targets but failed official warm-start replay',
 'closest_historical_failure_case': 'sequence_trend_joint_warm_start_official_eval_v1 had zero full-gate candidates under official warm-start metrics',
 'why_not_duplicate': 'This notebook builds new fold-local feature-quantile target contracts with both consensus reentry floors and drawdown caps, then evaluates them through the official warm-start replay gate.',
 'changed_layer': [

In [5]:

def max_drawdown_interval_from_daily(path: Path, *, label: str, candidate: str, fold: str) -> dict[str, Any]:
    daily = pd.read_csv(path, parse_dates=["datetime"])
    nav = pd.to_numeric(daily["strategy_nav"], errors="coerce").ffill()
    peak = nav.cummax()
    dd = nav / peak - 1.0
    trough_idx = int(dd.idxmin()) if len(dd) else 0
    peak_idx = int(nav.loc[:trough_idx].idxmax()) if len(nav.loc[:trough_idx]) else trough_idx
    interval = daily.loc[peak_idx:trough_idx].copy()
    start_nav = float(nav.iloc[peak_idx]) if len(nav) else math.nan
    trough_nav = float(nav.iloc[trough_idx]) if len(nav) else math.nan
    interval_return = (trough_nav / start_nav - 1.0) * 100.0 if math.isfinite(start_nav) and start_nav else math.nan
    action_col = daily.get("action", pd.Series("", index=daily.index)).astype(str).str.upper()
    return {
        "label": label,
        "candidate": candidate,
        "fold": fold,
        "path": str(path.relative_to(ROOT)),
        "peak_date": str(pd.Timestamp(daily.loc[peak_idx, "datetime"]).date()) if len(daily) else "",
        "trough_date": str(pd.Timestamp(daily.loc[trough_idx, "datetime"]).date()) if len(daily) else "",
        "max_dd": float(abs(dd.min()) * 100.0) if len(dd) else math.nan,
        "interval_return": float(interval_return),
        "interval_days": int(len(interval)),
        "interval_avg_pos": float(pd.to_numeric(interval.get("position", pd.Series(0.0, index=interval.index)), errors="coerce").fillna(0.0).mean()) if len(interval) else math.nan,
        "interval_min_pos": float(pd.to_numeric(interval.get("position", pd.Series(0.0, index=interval.index)), errors="coerce").fillna(0.0).min()) if len(interval) else math.nan,
        "interval_max_pos": float(pd.to_numeric(interval.get("position", pd.Series(0.0, index=interval.index)), errors="coerce").fillna(0.0).max()) if len(interval) else math.nan,
        "interval_trades": int(action_col.loc[peak_idx:trough_idx].isin(["BUY", "SELL"]).sum()) if len(daily) else 0,
    }

selection = json.loads((CARD / "selection.json").read_text(encoding="utf-8"))
summary = json.loads((CARD / "summary.json").read_text(encoding="utf-8"))
selected_candidate = selection.get("selected_candidate")
diag_rows: list[dict[str, Any]] = []
for fold in WALK_FORWARD_WINDOWS:
    candidate_path = CARD / f"{selected_candidate}_{fold}_warm_start_official_daily_path.csv"
    if candidate_path.exists():
        diag_rows.append(max_drawdown_interval_from_daily(candidate_path, label="selected", candidate=str(selected_candidate), fold=fold))
    ref_path = WARM_OFFICIAL_CARD / f"trend_plain_contract_{fold}_warm_start_official_daily_path.csv"
    if ref_path.exists():
        diag_rows.append(max_drawdown_interval_from_daily(ref_path, label="trend_plain_official_reference", candidate="trend_plain_contract", fold=fold))

drawdown_diag = pd.DataFrame(diag_rows).sort_values(["fold", "label"]).reset_index(drop=True)
drawdown_diag.to_csv(CARD / "drawdown_interval_diagnostic.csv", index=False)
summary["drawdown_interval_diagnostic"] = drawdown_diag.to_dict("records")
if "drawdown_interval_diagnostic.csv" not in summary.get("sidecars", []):
    summary.setdefault("sidecars", []).append("drawdown_interval_diagnostic.csv")
if "drawdown_interval_diagnostic.csv" not in selection.get("sidecars", []):
    selection.setdefault("sidecars", []).append("drawdown_interval_diagnostic.csv")

selected_wf2022 = drawdown_diag[(drawdown_diag["label"].eq("selected")) & (drawdown_diag["fold"].eq("wf_2022"))]
ref_wf2022 = drawdown_diag[(drawdown_diag["label"].eq("trend_plain_official_reference")) & (drawdown_diag["fold"].eq("wf_2022"))]
if not selected_wf2022.empty and not ref_wf2022.empty:
    sel = selected_wf2022.iloc[0]
    ref = ref_wf2022.iloc[0]
    diag_line = (
        f"Selected candidate's wf_2022 max-drawdown interval runs {sel['peak_date']} to {sel['trough_date']} "
        f"with avg position {sel['interval_avg_pos']:.3f} and MaxDD {sel['max_dd']:.2f}%, versus trend_plain reference "
        f"MaxDD {ref['max_dd']:.2f}% and avg position {ref['interval_avg_pos']:.3f}; risk-state labels did not de-risk early enough after increasing reentry participation."
    )
    if diag_line not in summary["conclusion"]:
        insert_at = max(0, len(summary["conclusion"]) - 1)
        summary["conclusion"].insert(insert_at, diag_line)
selection["drawdown_diagnostic"] = {
    "selected_candidate": selected_candidate,
    "diagnostic_csv": "drawdown_interval_diagnostic.csv",
    "interpretation": "reentry participation improved relative return, but drawdown interval exposure rose; next work should improve risk-state representation/checkpoint scoring before raising floors.",
}
write_json(CARD / "selection.json", selection)
write_json(CARD / "summary.json", summary)
display(drawdown_diag)


,label,candidate,fold,path,peak_date,trough_date,max_dd,interval_return,interval_days,interval_avg_pos,interval_min_pos,interval_max_pos,interval_trades
0,selected,multi_obj_floor45_cap20_riskfirst,wf_2022,runtime/model_selection/card/state_conditioned...,2022-01-03,2022-11-09,30.626799,-30.626799,216,0.131131,0.048456,0.450000,28
1,trend_plain_official_reference,trend_plain_contract,wf_2022,runtime/model_selection/card/sequence_trend_jo...,2022-01-03,2022-11-03,20.198957,-20.198957,212,0.114577,0.000000,0.457096,28
2,selected,multi_obj_floor45_cap20_riskfirst,wf_2023_2024,runtime/model_selection/card/state_conditioned...,2024-03-07,2024-04-19,16.122029,-16.122029,31,0.222731,0.095961,0.512402,6
3,trend_plain_official_reference,trend_plain_contract,wf_2023_2024,runtime/model_selection/card/sequence_trend_jo...,2024-07-10,2024-12-19,12.285728,-12.285728,115,0.110407,0.044066,0.214593,15


In [6]:

selection = json.loads((CARD / "selection.json").read_text(encoding="utf-8"))
summary = json.loads((CARD / "summary.json").read_text(encoding="utf-8"))
selected = selection.get("selected_summary", {})
lines = [
    "## Runtime Conclusion",
    "",
    f"Decision: `{selection['decision']}`. Promotion is blocked.",
    "",
    f"Selected candidate: `{selection.get('selected_candidate')}`.",
    "",
    f"Selected aggregate: min relative `{selected.get('min_relative_return')}`, max MaxDD `{selected.get('max_max_dd')}`, full-gate count `{selected.get('full_gate_count')}`.",
    "",
    "Key findings:",
]
lines.extend(f"- {line}" for line in summary.get("conclusion", []))
lines.extend([
    "",
    "Main sidecars:",
    "- `warm_start_official_metrics.csv`",
    "- `candidate_summary.csv`",
    "- `target_state_audit.csv`",
    "- `bc_slice_metrics.csv`",
    "- `selection.json` / `summary.json`",
    "",
    f"Next action: `{selection['next_mechanism']}`.",
])
display(Markdown("\n".join(lines)))


## Runtime Conclusion

Decision: `state_conditioned_multi_objective_drawdown_failed_no_promotion`. Promotion is blocked.

Selected candidate: `multi_obj_floor45_cap20_riskfirst`.

Selected aggregate: min relative `30.262520526732697`, max MaxDD `30.626798672198042`, full-gate count `0`.

Key findings:
- multi_obj_floor45_cap20_riskfirst: min official relative=30.26pp, max MaxDD=30.63%, full_gate_all=False, gate_loss=167.49.
- multi_obj_floor65_cap20_weighted: min official relative=22.62pp, max MaxDD=31.57%, full_gate_all=False, gate_loss=212.55.
- multi_obj_floor55_cap25_balanced: min official relative=8.06pp, max MaxDD=31.81%, full_gate_all=False, gate_loss=230.39.
- No candidate clears the +30pp / <=15% official warm-start gate across both folds; no promotion, no actor retention, and no long PPO from this card.
- Selected candidate's wf_2022 max-drawdown interval runs 2022-01-03 to 2022-11-09 with avg position 0.131 and MaxDD 30.63%, versus trend_plain reference MaxDD 20.20% and avg position 0.115; risk-state labels did not de-risk early enough after increasing reentry participation.
- Next action: revise risk-state representation or checkpoint scoring before increasing reentry floor.

Main sidecars:
- `warm_start_official_metrics.csv`
- `candidate_summary.csv`
- `target_state_audit.csv`
- `bc_slice_metrics.csv`
- `selection.json` / `summary.json`

Next action: `revise risk-state representation or checkpoint scoring before increasing reentry floor`.